# Inspect LangExtract: Per-Type Semantic Enrichment

Before running batch enrichment (step 4), this notebook demonstrates [LangExtract](https://github.com/google/langextract) on one chunk from each data type — Reddit, Zoom, and PDF. It shows how to design extraction schemas with few-shot examples, run extraction with Vertex AI auth, inspect the source-grounded results, and aggregate extractions into flat tag lists (the format stored in BigQuery).

LangExtract uses LLMs to extract structured entities from text with source grounding — each extraction points back to the exact text span it was derived from.

---
## Setup

In [1]:
import langextract as lx
from google.cloud import bigquery
from config import PROJECT_ID, BQ_PROJECT, BQ_DATASET, BQ_TABLE_PREFIX, FAST_GEMINI_MODEL

bq = bigquery.Client(project=BQ_PROJECT)

# LangExtract Vertex AI auth params (reused across all extractions)
lm_params = {"vertexai": True, "project": PROJECT_ID, "location": "global"}

print(f'Project: {PROJECT_ID}')
print(f'Model: {FAST_GEMINI_MODEL}')
print(f'Dataset: {BQ_PROJECT}.{BQ_DATASET}')

Project: statmike-mlops-349915
Model: gemini-2.5-flash
Dataset: statmike-mlops-349915.applied_genai_solutions


---
## Read Sample Chunks

Pull one chunk from each type to test extraction schemas on representative content.

In [2]:
# One chunk per type
reddit_row = list(bq.query(
    f"SELECT chunk_id, text_content FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE_PREFIX}_reddit_chunks` LIMIT 1"
).result())[0]

zoom_row = list(bq.query(
    f"SELECT chunk_id, text_content FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE_PREFIX}_zoom_chunks` LIMIT 1"
).result())[0]

pdf_row = list(bq.query(
    f"SELECT chunk_id, text_content FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE_PREFIX}_pdf_chunks` LIMIT 1"
).result())[0]

print(f'Reddit: {reddit_row.chunk_id} ({len(reddit_row.text_content)} chars)')
print(f'Zoom:   {zoom_row.chunk_id} ({len(zoom_row.text_content)} chars)')
print(f'PDF:    {pdf_row.chunk_id} ({len(pdf_row.text_content)} chars)')

Reddit: red_thread_deep_000 (642 chars)
Zoom:   zoom_meeting_business_readout_002 (2250 chars)
PDF:    pdf_2f59757db2_c1 (1747 chars)


---
## Reddit Extraction

Reddit chunks contain threaded discussions about forecasting methods. Extract two classes:
- **`topic`** — data science topics mentioned (e.g., "demand forecasting", "holiday effects")
- **`method`** — specific forecasting methods or tools (e.g., "Prophet", "ARIMA")

In [3]:
reddit_prompt = "Extract forecasting methods and data science topics mentioned. Use exact text for extractions."

reddit_examples = [
    lx.data.ExampleData(
        text="THREAD: ARIMA vs. Prophet | COMMENT: I've had great results with Prophet for seasonal retail demand. It handles holiday effects automatically.",
        extractions=[
            lx.data.Extraction(extraction_class="method", extraction_text="Prophet",
                attributes={"name": "Prophet"}),
            lx.data.Extraction(extraction_class="topic", extraction_text="seasonal retail demand",
                attributes={"name": "demand forecasting"}),
            lx.data.Extraction(extraction_class="topic", extraction_text="holiday effects",
                attributes={"name": "holiday effects"}),
        ]
    )
]

print(f'Extracting from: {reddit_row.chunk_id}')
print(f'Text preview: {reddit_row.text_content[:200]}...\n')

reddit_result = lx.extract(
    text_or_documents=reddit_row.text_content,
    prompt_description=reddit_prompt,
    examples=reddit_examples,
    model_id=FAST_GEMINI_MODEL,
    language_model_params=lm_params,
)

print(f'{len(reddit_result.extractions)} extractions found:\n')
for e in reddit_result.extractions:
    print(f'  [{e.extraction_class}] "{e.extraction_text}" -> {e.attributes}')

Extracting from: red_thread_deep_000
Text preview: THREAD: ARIMA vs. Prophet vs. LightGBM for Demand Forecasting - What's your go-to and why? | COMMENT: For data with clear seasonal patterns, a well-specified SARIMA model is hard to beat. The key is r...



LangExtract: model=gemini-2.5-flash, current=642 chars, processed=0 chars:  [00:05]

16 extractions found:

  [method] "ARIMA" -> {'name': 'ARIMA'}
  [method] "Prophet" -> {'name': 'Prophet'}
  [method] "LightGBM" -> {'name': 'LightGBM'}
  [topic] "Demand Forecasting" -> {'name': 'demand forecasting'}
  [topic] "seasonal patterns" -> {'name': 'seasonal patterns'}
  [method] "SARIMA model" -> {'name': 'SARIMA'}
  [topic] "diagnostics" -> {'name': 'diagnostics'}
  [topic] "stationary" -> {'name': 'stationarity'}
  [topic] "ADF test" -> {'name': 'ADF test'}
  [topic] "differencing" -> {'name': 'differencing'}
  [topic] "ACF" -> {'name': 'ACF'}
  [topic] "PACF" -> {'name': 'PACF'}
  [topic] "p, d, q parameters" -> {'name': 'ARIMA parameters'}
  [topic] "P, D, Q parameters" -> {'name': 'SARIMA parameters'}
  [topic] "uncertainty intervals" -> {'name': 'uncertainty intervals'}
  [topic] "statistical rigor" -> {'name': 'statistical rigor'}


---
## Zoom Extraction

Zoom chunks contain meeting transcript segments with speaker labels. Extract two classes:
- **`topic`** — discussion topics (e.g., "model comparison", "data quality")
- **`action_item`** — action items or decisions with owners

In [4]:
zoom_prompt = "Extract discussion topics and action items or decisions from meeting transcript chunks."

zoom_examples = [
    lx.data.ExampleData(
        text="Sarah (Lead): We need to benchmark ARIMA against Prophet on the Q4 data. David, can you set that up?\nDavid (Statistician): Sure, I'll have results by Friday.",
        extractions=[
            lx.data.Extraction(extraction_class="topic", extraction_text="benchmark ARIMA against Prophet",
                attributes={"name": "model comparison"}),
            lx.data.Extraction(extraction_class="action_item", extraction_text="I'll have results by Friday",
                attributes={"description": "Benchmark ARIMA vs Prophet on Q4 data", "owner": "David"}),
        ]
    )
]

print(f'Extracting from: {zoom_row.chunk_id}')
print(f'Text preview: {zoom_row.text_content[:200]}...\n')

zoom_result = lx.extract(
    text_or_documents=zoom_row.text_content,
    prompt_description=zoom_prompt,
    examples=zoom_examples,
    model_id=FAST_GEMINI_MODEL,
    language_model_params=lm_params,
)

print(f'{len(zoom_result.extractions)} extractions found:\n')
for e in zoom_result.extractions:
    print(f'  [{e.extraction_class}] "{e.extraction_text}" -> {e.attributes}')

Extracting from: zoom_meeting_business_readout_002
Text preview: [Summary: The team agrees on a consensus forecast allowing for auditable manual adjustments.] Maria Rodriguez: I like that. A consensus forecast. If we can adjust the enterprise segment projection up ...



LangExtract: model=gemini-2.5-flash, current=2,248 chars, processed=0 chars:  [00:19]

22 extractions found:

  [topic] "Consensus forecast with auditable manual adjustments" -> {'name': 'Consensus forecasting'}
  [action_item] "Anya's team will continue with the model" -> {'description': "Anya's team to continue using the current forecasting model", 'owner': "Anya's team"}
  [action_item] "Add a consensus review meeting to the quarterly process" -> {'description': 'Integrate a consensus review meeting into the regular quarterly forecasting process', 'owner': 'Tom'}
  [action_item] "Schedule consensus review meeting for early next week" -> {'description': 'Schedule the first consensus review meeting for the current quarter', 'owner': 'Tom'}
  [topic] "Category-level breakdown and historical SKU-mix percentages" -> None
  [action_item] "We'll come prepared with the category-level breakdown and the historical SKU-mix percentages" -> {'description': 'Prepare category-level breakdown and historical SKU-mix percentages', 'owner': 'Dr. Anya Sharma'}
  [topic] "Impact of higher

---
## PDF Extraction

PDF chunks contain BigQuery ML documentation. Extract two classes:
- **`topic`** — technical concepts (e.g., "time series modeling", "seasonality detection")
- **`sql_function`** — BigQuery ML functions (e.g., "CREATE MODEL", "ML.FORECAST")

In [5]:
pdf_prompt = "Extract BigQuery ML functions and technical concepts from documentation chunks."

pdf_examples = [
    lx.data.ExampleData(
        text="# The CREATE MODEL statement for ARIMA PLUS\n\nUse the CREATE MODEL statement to create a time series model with automatic seasonality detection.",
        extractions=[
            lx.data.Extraction(extraction_class="sql_function", extraction_text="CREATE MODEL",
                attributes={"name": "CREATE MODEL"}),
            lx.data.Extraction(extraction_class="topic", extraction_text="time series model",
                attributes={"name": "time series modeling"}),
            lx.data.Extraction(extraction_class="topic", extraction_text="automatic seasonality detection",
                attributes={"name": "seasonality detection"}),
        ]
    )
]

print(f'Extracting from: {pdf_row.chunk_id}')
print(f'Text preview: {pdf_row.text_content[:200]}...\n')

pdf_result = lx.extract(
    text_or_documents=pdf_row.text_content,
    prompt_description=pdf_prompt,
    examples=pdf_examples,
    model_id=FAST_GEMINI_MODEL,
    language_model_params=lm_params,
)

print(f'{len(pdf_result.extractions)} extractions found:\n')
for e in pdf_result.extractions:
    print(f'  [{e.extraction_class}] "{e.extraction_text}" -> {e.attributes}')

Extracting from: pdf_2f59757db2_c1
Text preview: # The ML.ARIMA EVALUATE function

This document describes the ML.ARIMA_EVALUATE function, which you can use to evaluate the model metrics of ARIMA_PLUS or ARIMA_PLUS_XREG time series models.

## Synta...



LangExtract: model=gemini-2.5-flash, current=1,746 chars, processed=0 chars:  [00:05]

20 extractions found:

  [sql_function] "ML.ARIMA_EVALUATE" -> {'name': 'ML.ARIMA_EVALUATE'}
  [topic] "model metrics" -> {'name': 'model metrics'}
  [topic] "ARIMA_PLUS" -> {'name': 'ARIMA_PLUS'}
  [topic] "ARIMA_PLUS_XREG" -> {'name': 'ARIMA_PLUS_XREG'}
  [topic] "time series models" -> {'name': 'time series modeling'}
  [topic] "candidate models" -> {'name': 'candidate models'}
  [topic] "Akaike information criterion (AIC)" -> {'name': 'Akaike information criterion (AIC)'}
  [topic] "metrics" -> {'name': 'metrics'}
  [topic] "candidate models" -> {'name': 'candidate models'}
  [topic] "fitting error" -> {'name': 'fitting error'}
  [topic] "non_seasonal_p" -> {'name': 'non_seasonal_p'}
  [topic] "non_seasonal_d" -> {'name': 'non_seasonal_d'}
  [topic] "non_seasonal_q" -> {'name': 'non_seasonal_q'}
  [topic] "drift" -> {'name': 'drift'}
  [topic] "single time series training" -> {'name': 'single time series training'}
  [topic] "auto.ARIMA" -> {'name': 'auto.ARIMA'}
  [topic] "large-s

---
## Compare Results

Side-by-side view of extractions from all three types. Notice how extraction schemas differ by content type — Reddit yields methods and discussion topics, Zoom yields action items and meeting topics, and PDF yields SQL functions and technical concepts.

In [6]:
import pandas as pd

rows = []
for source, result in [('reddit', reddit_result), ('zoom', zoom_result), ('pdf', pdf_result)]:
    for e in result.extractions:
        rows.append({
            'source_type': source,
            'class': e.extraction_class,
            'extraction_text': e.extraction_text[:80],
            'attributes': str(e.attributes),
        })

df = pd.DataFrame(rows)
print(f'{len(df)} total extractions across all types\n')
df

58 total extractions across all types



,source_type,class,extraction_text,attributes
0,reddit,method,ARIMA,{'name': 'ARIMA'}
1,reddit,method,Prophet,{'name': 'Prophet'}
2,reddit,method,LightGBM,{'name': 'LightGBM'}
3,reddit,topic,Demand Forecasting,{'name': 'demand forecasting'}
4,reddit,topic,seasonal patterns,{'name': 'seasonal patterns'}
5,reddit,method,SARIMA model,{'name': 'SARIMA'}
6,reddit,topic,diagnostics,{'name': 'diagnostics'}
7,reddit,topic,stationary,{'name': 'stationarity'}
8,reddit,topic,ADF test,{'name': 'ADF test'}
9,reddit,topic,differencing,{'name': 'differencing'}


---
## Aggregate to Tags

The enrichment scripts (step 4a-4c) store extractions as flat lists in BigQuery REPEATED STRING columns. This cell shows the aggregation logic: group extractions by class and collect the `name` attribute (or `description` for action items) into deduplicated lists.

In [7]:
def aggregate_tags(result, class_to_field):
    """Aggregate extractions into tag lists by class.
    
    Args:
        result: LangExtract result with .extractions
        class_to_field: dict mapping extraction_class -> (column_name, attribute_key)
    
    Returns:
        dict mapping column_name -> list of unique tag strings
    """
    tags = {col: [] for col, _ in class_to_field.values()}
    for e in result.extractions:
        if e.extraction_class in class_to_field:
            col, attr_key = class_to_field[e.extraction_class]
            attrs = e.attributes or {}
            val = attrs.get(attr_key, e.extraction_text)
            if val and val not in tags[col]:
                tags[col].append(val)
    return tags


# Reddit: topics + methods_mentioned
reddit_tags = aggregate_tags(reddit_result, {
    'topic': ('topics', 'name'),
    'method': ('methods_mentioned', 'name'),
})
print(f'Reddit tags:')
for col, vals in reddit_tags.items():
    print(f'  {col}: {vals}')

# Zoom: topics + action_items
zoom_tags = aggregate_tags(zoom_result, {
    'topic': ('topics', 'name'),
    'action_item': ('action_items', 'description'),
})
print(f'\nZoom tags:')
for col, vals in zoom_tags.items():
    print(f'  {col}: {vals}')

# PDF: topics + functions_referenced
pdf_tags = aggregate_tags(pdf_result, {
    'topic': ('topics', 'name'),
    'sql_function': ('functions_referenced', 'name'),
})
print(f'\nPDF tags:')
for col, vals in pdf_tags.items():
    print(f'  {col}: {vals}')

Reddit tags:
  topics: ['demand forecasting', 'seasonal patterns', 'diagnostics', 'stationarity', 'ADF test', 'differencing', 'ACF', 'PACF', 'ARIMA parameters', 'SARIMA parameters', 'uncertainty intervals', 'statistical rigor']
  methods_mentioned: ['ARIMA', 'Prophet', 'LightGBM', 'SARIMA']

Zoom tags:
  topics: ['Consensus forecasting', 'Category-level breakdown and historical SKU-mix percentages', 'Impact of higher close rate on enterprise deals', 'Reporting dashboard requirements', 'data quality', 'Data Quality', 'null', 'Meeting Logistics']
  action_items: ["Anya's team to continue using the current forecasting model", 'Integrate a consensus review meeting into the regular quarterly forecasting process', 'Schedule the first consensus review meeting for the current quarter', 'Prepare category-level breakdown and historical SKU-mix percentages', 'Run a simulation for the impact of a 5% higher close rate on enterprise deals', 'Ensure final output includes base model forecast, consensu